# AI Coloring Book — Final Prototype

This notebook runs the complete **name → grounded biography → FLUX line art → PDF book** pipeline. The default `T4_SAFE_MODE` is designed for a free Colab **T4 GPU**; an L4 remains faster when available. Stage outputs are cached, so rerunning the final cell resumes incomplete work.


In [ ]:
# Confirm that Colab assigned a GPU. Free T4 and paid L4 runtimes are supported.
!nvidia-smi

In [ ]:
from pathlib import Path

REPOSITORY = 'https://github.com/icynic/AI-coloring-book.git'
PROJECT_DIR = Path('/content/AI-coloring-book')
if not PROJECT_DIR.exists():
    !git clone -q {REPOSITORY} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull --ff-only
%cd /content/AI-coloring-book
!git rev-parse --short HEAD

In [ ]:
# Install the final-prototype environment. Pillow is kept within a compatible
# range to avoid replacing Colab's already-loaded PIL modules.
%pip install -q -r requirements-colab.txt

In [ ]:
# Fail early if pip left an incompatible or mixed Colab environment.
from importlib.metadata import version as package_version
from packaging.version import Version

try:
    import PIL
    from PIL import Image, ImageOps
    print('Pillow compatibility check passed:', PIL.__version__)
except ImportError as exc:
    raise RuntimeError(
        'Pillow core modules could not be imported. Re-run the dependency '
        'installation cell, choose Runtime > Restart session, and then run '
        'the notebook again.'
    ) from exc

requests_version = package_version('requests')
protobuf_version = Version(package_version('protobuf'))
if requests_version != '2.32.4' or not Version('5.29.1') <= protobuf_version < Version('6'):
    raise RuntimeError(
        f'Incompatible Colab dependencies: requests={requests_version}, '
        f'protobuf={protobuf_version}. Re-run the install cell using the updated '
        'requirements-colab.txt, then restart the session.'
    )
print('Colab dependency check passed:', requests_version, protobuf_version)

## Configure the run
The default names are the frozen Marburg evaluation set. The models are loaded sequentially, not simultaneously. `T4_SAFE_MODE=True` applies Qwen 4-bit, FLUX 8-bit, 640px output, and a shorter prompt sequence. Google Drive is recommended because Colab runtimes can disconnect. Set `FORCE_REGENERATE=True` only when you deliberately want to overwrite cached stage outputs.

In [ ]:
NAMES = [
    line.strip()
    for line in Path('evaluation/subjects.txt').read_text(encoding='utf-8').splitlines()
    if line.strip()
]

USE_GOOGLE_DRIVE = True
T4_SAFE_MODE = True          # recommended for the free 16GB T4
FORCE_REGENERATE = False
FUZZY_SEARCH = False       # the frozen evaluation titles are exact
SEED = 42
QWEN_QUANTIZATION = 'none'   # used only when T4_SAFE_MODE is False
FLUX_QUANTIZATION = 'none'   # used only when T4_SAFE_MODE is False
FLUX_OFFLOAD = False         # used only when T4_SAFE_MODE is False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/AIColoringBook/evaluation_flux_t4'
else:
    OUTPUT_DIR = '/content/AIColoringBook/evaluation_flux_t4'

print('Output directory:', OUTPUT_DIR)
print('T4 safe mode:', T4_SAFE_MODE)

In [ ]:
# Run or resume the complete pipeline.
from main import main as run_coloring_book

arguments = [
    '--names', *NAMES,
    '--output-dir', OUTPUT_DIR,
    '--seed', str(SEED),
]
if not FUZZY_SEARCH:
    arguments.append('--no-fuzzy-search')
if T4_SAFE_MODE:
    arguments.append('--t4-safe-mode')
else:
    arguments.extend([
        '--qwen-quantization', QWEN_QUANTIZATION,
        '--flux-quantization', FLUX_QUANTIZATION,
    ])
    if FLUX_OFFLOAD:
        arguments.append('--flux-offload')
if FORCE_REGENERATE:
    arguments.append('--force')

run_coloring_book(arguments)

In [ ]:
# Inspect the reproducibility record and display the finished book link.
import json
from IPython.display import FileLink, display

manifest_path = Path(OUTPUT_DIR) / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps(manifest['runtime'], indent=2))
for item in manifest['items']:
    print(item['title'], 'OK' if not item['errors'] else item['errors'])

book_path = Path(OUTPUT_DIR) / 'coloring_book.pdf'
display(FileLink(str(book_path)))